# Radiology Assistant: End-to-End Evaluation (Zip-Ready)

### Instructions for Colab:
1. **Upload your Zip**: Upload your project zip file to the Colab `/content` folder (using the folder icon on the left).
2. **Set Secrets**: Click the **🔑 (Secrets)** icon and add:
   - `OPENAI_API_KEY`, `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.
3. **Run Cells**: The first code cell will automatically find and unzip your project.

In [1]:
# 1. Unzip and Robust Path Setup
import os, sys, zipfile, shutil
from pathlib import Path

# Find the uploaded zip file in /content
zips = list(Path('/content').glob('*.zip'))
if not zips:
    print("No zip file found in /content. Assuming files are already present.")
    REPO_ROOT = Path.cwd()
else:
    zip_path = zips[0]
    extract_to = Path('/content/extracted_project')
    print(f"Unzipping {zip_path.name} to {extract_to}...")
    if extract_to.exists(): shutil.rmtree(extract_to)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    # Robustly find the directory containing 'src'
    # This handles nested folders or __MACOSX garbage
    src_dirs = list(extract_to.rglob('src'))
    if src_dirs:
        # The parent of the 'src' folder is our REPO_ROOT
        REPO_ROOT = src_dirs[0].parent
        print(f"Detected REPO_ROOT at: {REPO_ROOT}")
    else:
        REPO_ROOT = extract_to
        print(f"Warning: 'src' folder not found. Using {REPO_ROOT} as root.")

# CRITICAL: Add the 'src' directory to sys.path so 'bda_chest' can be found
SRC_PATH = str(REPO_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Python path updated. Searching for modules in: {SRC_PATH}")

# Verify bda_chest exists
if (REPO_ROOT / "src" / "bda_chest").exists():
    print("✅ Found 'bda_chest' package.")
else:
    print("❌ ERROR: 'bda_chest' package not found in expected location.")

# Install Dependencies
!pip install -q "transformers<5.0.0" "accelerate>=0.25" "bitsandbytes>=0.43.0" "evaluate>=0.4.0" "rouge_score>=0.1.2" kaggle python-dotenv openai Pillow timm

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Unzipping Big_Data_Analytics_Midterm_Project.zip to /content/extracted_project...
Detected REPO_ROOT at: /content/extracted_project/Big_Data_Analytics_Midterm_Project
Python path updated. Searching for modules in: /content/extracted_project/Big_Data_Analytics_Midterm_Project/src
✅ Found 'bda_chest' package.


In [2]:
# 2. Import Your Trained Pipeline
try:
    from bda_chest.pipeline import load_inference_bundle, infer_from_pil
    from bda_chest.llm import answer_question_about_report
    print("✅ Successfully imported project modules.")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print("Current sys.path:", sys.path)
    raise

# Find checkpoint
CHECKPOINT_PATH = REPO_ROOT / "eva_x_tiny_binary_best.pt"
if not CHECKPOINT_PATH.exists():
    # Try alternative locations if it's missing from root
    alt_path = list(REPO_ROOT.rglob("eva_x_tiny_binary_best.pt"))
    if alt_path:
        CHECKPOINT_PATH = alt_path[0]
    else:
        raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT_PATH}.")

bundle = load_inference_bundle(CHECKPOINT_PATH, device_hint="cuda")
print(f"Trained EVA-X model loaded from {CHECKPOINT_PATH}")

✅ Successfully imported project modules.
Trained EVA-X model loaded from /content/extracted_project/Big_Data_Analytics_Midterm_Project/eva_x_tiny_binary_best.pt


In [3]:
# 3. Define Evaluation Judge (Gemma 2-2B)
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class MedGemmaJudge:
    def __init__(self, model_id="google/gemma-2-2b-it"):
        print(f"Loading Judge: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)

        if torch.cuda.is_available():
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                device_map="auto",
                attn_implementation="eager"
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                device_map="cpu",
                torch_dtype=torch.float32
            )
        self.model.eval()

    def evaluate(self, q, a, c, gt):
        # Medical Persona Prompt
        prompt = f"You are a senior radiologist evaluator. Assess the AI's answer for clinical accuracy.\n\nContext: {str(c)[:500]}\nQuestion: {q}\nAnswer: {str(a)[:500]}\nGround Truth: {gt}\n\nTask: Return a JSON object with:\n- \"correctness_score\": an integer 1-5\n- \"justification\": a brief clinical explanation.\n\nResponse:"

        messages = [{"role": "user", "content": prompt}]
        text_input = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = self.tokenizer(text_input, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.2,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [4]:
# 4. Download Kaggle Test Data
from kaggle.api.kaggle_api_extended import KaggleApi
import random

api = KaggleApi(); api.authenticate()
test_img_path = Path("/content/eval_test_images")
if not test_img_path.exists():
    api.dataset_download_files("paultimothymooney/chest-xray-pneumonia", path="/content/temp_kaggle", unzip=True)
    for cat in ["NORMAL", "PNEUMONIA"]:
        dest = test_img_path / cat.lower(); dest.mkdir(parents=True, exist_ok=True)
        imgs = list(Path("/content/temp_kaggle/chest_xray/test").joinpath(cat).glob("*.jpeg"))
        for i in random.sample(imgs, 3): shutil.copy(i, dest / i.name)
    shutil.rmtree("/content/temp_kaggle")
print("Test images ready at /content/eval_test_images")

Test images ready at /content/eval_test_images


In [5]:
# 5. Comprehensive Evaluation Loop
from PIL import Image
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Ensure judge is initialized (Restart session if CUDA error persists)
try:
    if 'judge' not in globals():
        judge = MedGemmaJudge()
except Exception as e:
    print(f"Could not initialize judge: {e}")

# Configuration
CONFIDENCE_THRESHOLD = 0.5
results = []

print("Starting Evaluation...")
test_files = list(test_img_path.rglob("*.jpeg"))

for i, path in enumerate(test_files):
    try:
        img = Image.open(path).convert("RGB")
        ground_truth_label = path.parent.name.upper() # NORMAL or PNEUMONIA
        q = "Are there signs of pneumonia?"

        # --- Stage 1: Classification ---
        payload, prob = infer_from_pil(bundle, img, threshold=CONFIDENCE_THRESHOLD)
        prediction_label = payload['prediction']

        # --- Stage 2: QA / Reasoning ---
        answer = answer_question_about_report(payload, q)

        # --- Stage 3: Clinical Critique (Judge) ---
        # Truncate inputs to prevent OOM/Asserts
        feedback = judge.evaluate(q, answer, payload['impression'][:500], f"Verify if the report matches the diagnosis of {ground_truth_label}.")

        # Parse feedback for score (simple heuristic)
        score = 0
        if "correctness_score" in feedback:
            try:
                # extract number from JSON-like string
                import re
                match = re.search(r'"correctness_score":\s*(\d)', feedback)
                if match:
                    score = int(match.group(1))
            except: pass

        # Store Result
        results.append({
            "filename": path.name,
            "ground_truth": ground_truth_label,
            "prediction": prediction_label,
            "probability": prob,
            "assistant_answer": answer,
            "judge_feedback": feedback,
            "judge_score": score
        })

        print(f"\n[{i+1}/{len(test_files)}] {path.name}")
        print(f"   GT: {ground_truth_label} | Pred: {prediction_label} ({prob:.4f})")
        print(f"   Judge: {feedback[:100]}...")

    except Exception as e:
        print(f"Error processing {path.name}: {e}")
        if "device-side assert" in str(e).lower():
            print("CRITICAL: GPU state corrupted. Please Restart Session.")
            break

# --- Metrics Calculation ---
if results:
    df = pd.DataFrame(results)

    # Binary Metrics
    y_true = df['ground_truth']
    y_pred = df['prediction']

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label='PNEUMONIA')

    print("\n" + "="*30)
    print("📊 FINAL EVALUATION REPORT")
    print("="*30)
    print(f"Total Samples: {len(df)}")
    print(f"Accuracy:      {acc:.4f}")
    print(f"Precision:     {prec:.4f}")
    print(f"Recall:        {rec:.4f}")
    print(f"F1 Score:      {f1:.4f}")

    print("\n--- Detailed Results ---")
    display(df[['filename', 'ground_truth', 'prediction', 'judge_score']])
else:
    print("No results collected.")

Loading google/gemma-2-2b-it...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Vocab Limit set to: 256000
Starting Evaluation...

[1/6] NORMAL2-IM-0259-0001.jpeg
   GT: NORMAL | Pred: PNEUMONIA (0.9910)
   Judge: ```json
{
  "correctness_score": 1,
  "justification": "The AI's answer is incorrect. The ground tru...

[2/6] NORMAL2-IM-0256-0001.jpeg
   GT: NORMAL | Pred: PNEUMONIA (0.9991)
   Judge: ```json
{
  "correctness_score": 1,
  "justification": "The AI's answer is incorrect. The ground tru...

[3/6] NORMAL2-IM-0361-0001.jpeg
   GT: NORMAL | Pred: NORMAL (0.2125)
   Judge: ```json
{
  "correctness_score": 5,
  "justification": "The AI correctly identifies that there are n...

[4/6] person134_bacteria_644.jpeg
   GT: PNEUMONIA | Pred: PNEUMONIA (0.9990)
   Judge: ```json
{
  "correctness_score": 4,
  "justification": "The AI's answer is technically correct in th...

[5/6] person109_bacteria_519.jpeg
   GT: PNEUMONIA | Pred: PNEUMONIA (0.9990)
   Judge: ```json
{
  "correctness_score": 4,
  "justification": "The AI correctly identifies a high-confidenc...

[6

,filename,ground_truth,prediction,judge_score
0,NORMAL2-IM-0259-0001.jpeg,NORMAL,PNEUMONIA,1
1,NORMAL2-IM-0256-0001.jpeg,NORMAL,PNEUMONIA,1
2,NORMAL2-IM-0361-0001.jpeg,NORMAL,NORMAL,5
3,person134_bacteria_644.jpeg,PNEUMONIA,PNEUMONIA,4
4,person109_bacteria_519.jpeg,PNEUMONIA,PNEUMONIA,4
5,person39_virus_85.jpeg,PNEUMONIA,PNEUMONIA,4
